# ODI to Databricks Migration: TRG_DEP Full Load

**Source File:** `TRG_DEP_Full_Load.txt`
**Conversion Timestamp:** 2024-07-30 12:00:00 UTC

This notebook performs a full load into the `TRG_DEP` table from the `DEPARTMENTS` source table.

In [ ]:
dbutils.widgets.text("ETL_JOB_TYPE", "", "1. ETL Job Type (e.g., 'F' for Full)")
dbutils.widgets.text("DATASOURCE_NUM_ID", "0", "2. Data Source Number ID")
dbutils.widgets.text("ETL_PROC_WID", "0", "3. ETL Process ID")
dbutils.widgets.text("ODI_SESS_NO", "0", "4. ODI Session Number")
dbutils.widgets.text("V_ETL_LAST_EXTRACT_TIME", "1900-01-01 00:00:00", "5. Last Extract Time (YYYY-MM-DD HH:MM:SS)")
dbutils.widgets.text("V_ETL_CURRENT_EXTRACT_TIME", "", "6. Current Extract Time (YYYY-MM-DD HH:MM:SS, defaults to current_timestamp() if empty)")

# ETL Parameters

These views provide access to the ETL parameters passed via widgets.

In [ ]:
%sql
-- Create temporary views for ETL parameters

CREATE OR REPLACE TEMPORARY VIEW v_etl_job_type AS
SELECT '${ETL_JOB_TYPE}' AS etl_job_type;

CREATE OR REPLACE TEMPORARY VIEW v_datasource_num_id AS
SELECT CAST(${DATASOURCE_NUM_ID} AS BIGINT) AS datasource_num_id;

CREATE OR REPLACE TEMPORARY VIEW v_etl_proc_wid AS
SELECT CAST(${ETL_PROC_WID} AS BIGINT) AS etl_proc_wid;

CREATE OR REPLACE TEMPORARY VIEW v_odi_sess_no AS
SELECT '${ODI_SESS_NO}' AS odi_sess_no;

CREATE OR REPLACE TEMPORARY VIEW v_etl_last_extract_time AS
SELECT to_timestamp('${V_ETL_LAST_EXTRACT_TIME}', 'yyyy-MM-dd HH:mm:ss') AS etl_last_extract_time;

CREATE OR REPLACE TEMPORARY VIEW v_etl_current_extract_time AS
SELECT COALESCE(to_timestamp(NULLIF('${V_ETL_CURRENT_EXTRACT_TIME}', ''), 'yyyy-MM-dd HH:mm:ss'), current_timestamp()) AS etl_current_extract_time;

In [ ]:
display(spark.sql("""
  SELECT
    (SELECT etl_job_type FROM v_etl_job_type) AS ETL_JOB_TYPE,
    (SELECT datasource_num_id FROM v_datasource_num_id) AS DATASOURCE_NUM_ID,
    (SELECT etl_proc_wid FROM v_etl_proc_wid) AS ETL_PROC_WID,
    (SELECT odi_sess_no FROM v_odi_sess_no) AS ODI_SESS_NO,
    (SELECT etl_last_extract_time FROM v_etl_last_extract_time) AS V_ETL_LAST_EXTRACT_TIME,
    (SELECT etl_current_extract_time FROM v_etl_current_extract_time) AS V_ETL_CURRENT_EXTRACT_TIME
"""))

# Target Table: TRG_DEP

Operations for the `TRG_DEP` target table.

In [ ]:
%sql
-- SCEN_TASK_NO {10}: Prepare Target TRG_DEP

-- Create target table if it does not exist
-- Inferred schema based on typical HR.DEPARTMENTS structure
CREATE TABLE IF NOT EXISTS workspace.hr.trg_dep (
    department_id   BIGINT,
    department_name STRING,
    manager_id      BIGINT,
    location_id     BIGINT
) USING DELTA;

In [ ]:
%sql
-- SCEN_TASK_NO {20}: Truncate Target Table (Full Load assumption)

TRUNCATE TABLE workspace.hr.trg_dep;

In [ ]:
%sql
-- SCEN_TASK_NO {30}: Insert into TRG_DEP
-- Oracle hints /*+ APPEND PARALLEL */ have been removed as they are not applicable to Delta Lake.

INSERT INTO workspace.hr.trg_dep
  (
    department_id ,
    department_name ,
    manager_id ,
    location_id
  )
SELECT
  departments.department_id ,
  departments.department_name ,
  departments.manager_id ,
  departments.location_id
FROM
  workspace.hr.departments AS departments;

In [ ]:
%sql
-- Record count after insert
SELECT COUNT(*) FROM workspace.hr.trg_dep;

# Optimize Target

Optimizing the Delta table for query performance.

In [ ]:
%sql
-- Disable ZORDER stats check to prevent DELTA_ZORDERING_ON_COLUMN_WITHOUT_STATS
SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;
OPTIMIZE workspace.hr.trg_dep ZORDER BY (department_id);

# Cleanup

This section would typically drop temporary staging or flow tables. No such tables were used in this conversion.

# Validation

Reviewing the loaded data.

In [ ]:
%sql
SELECT * FROM workspace.hr.trg_dep LIMIT 10;

# Conversion Notes

1.  **Schema and Table Naming**: Oracle schema `HR` converted to `workspace.hr`. Table names `TRG_DEP` and `DEPARTMENTS` converted to lowercase `trg_dep` and `departments`.
2.  **Oracle Hints**: The `/*+ APPEND PARALLEL */` hint has been removed as it is specific to Oracle and not applicable to Delta Lake.
3.  **Full Load Assumption**: Since the source only provided an `INSERT` statement for `TRG_DEP` without preceding `DELETE` or `TRUNCATE` logic, this conversion assumes a full load scenario, adding a `TRUNCATE TABLE` before the `INSERT` to clear existing data in the target. If this is an incremental append, the `TRUNCATE TABLE` statement should be removed.
4.  **Target Table DDL**: The `CREATE TABLE IF NOT EXISTS workspace.hr.trg_dep` statement includes an inferred schema based on typical Oracle HR.DEPARTMENTS table structure and Databricks data type mappings.
5.  **Optimization**: An `OPTIMIZE ... ZORDER BY` statement has been added for `department_id` to improve query performance on this key column. This is preceded by `SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;` as required.
6.  **ETL Parameters**: Standard ETL parameter widgets (`ETL_JOB_TYPE`, `DATASOURCE_NUM_ID`, `ETL_PROC_WID`, `ODI_SESS_NO`, `V_ETL_LAST_EXTRACT_TIME`, `V_ETL_CURRENT_EXTRACT_TIME`) have been included and made available via temporary views, even though they are not directly referenced in the simple INSERT logic of the original source. This follows the standard notebook template.